# 08 — Inverted-Mel mixture ratios

Trains mixed inverted-Mel at four epoch shares of PA: 100%, 75%, 25%, and 0%. The 50/50 checkpoint already in the paper table is not trained again.

Each epoch has the same length as that 50/50 run. The 2017 files are repeated when their share is larger than the 3,014 real training files. Seed 42 matches the speaker split of the existing run. Checkpoints go to `lfcc_vs_mel_compare/runs/inverted_mel_mix_paXXX/`.

Each new checkpoint is scored on 2017 eval and PA eval, then PA minimum t-DCF is computed. Expect about four hours.


In [1]:
import json
import sys
from pathlib import Path

HERE = Path.cwd()
if not (HERE / "common.py").exists():
    HERE = Path(r"D:\speaker-verification-system\replay-cnn-baseline\experiments\paper_official_eval")
sys.path.insert(0, str(HERE))

from common import asv17_root, ensure_pa_ready, experiments_dir, pa_root, paper_runs
from compute_pa_tdcf import min_tdcf_for_predictions

COMPARE = experiments_dir() / "lfcc_vs_mel_compare"
sys.path.insert(0, str(COMPARE))
from experiment_lib import eval_checkpoint_on_corpora, train_feature

ensure_pa_ready()

SEED42 = COMPARE / "runs" / "inverted_mel" / "best_inverted_mel_mixed_2017_pa2019.pt"
if not SEED42.is_file():
    raise FileNotFoundError(SEED42)

saved = json.loads(
    (paper_runs() / "mixed_frontends_eval" / "inverted_mel" / "metrics.json").read_text(encoding="utf-8")
)
saved_pa_scores = paper_runs() / "mixed_frontends_eval" / "inverted_mel" / "pa2019_predictions.csv"
rows = [{
    "pa_fraction": 0.5,
    "label": "50/50",
    "checkpoint": str(SEED42),
    "eer_2017": saved["asvspoof2017"]["eer_percent"],
    "n_2017": saved["asvspoof2017"]["n"],
    "eer_pa": saved["pa2019"]["eer_percent"],
    "n_pa": saved["pa2019"]["n"],
    "min_tdcf": min_tdcf_for_predictions(saved_pa_scores),
    "trained_now": False,
}]

RATIOS = (
    (1.0, "100% PA"),
    (0.75, "75/25"),
    (0.25, "25/75"),
    (0.0, "100% 2017"),
)
for fraction, label in RATIOS:
    tag = f"pa{int(round(fraction * 100)):03d}"
    out = COMPARE / "runs" / f"inverted_mel_mix_{tag}"
    print(f"\n===== train inverted_mel {label} -> {out} =====", flush=True)
    summary = train_feature(
        "inverted_mel",
        asv17_root=asv17_root(),
        pa_root=pa_root(),
        output_dir=out,
        epochs=15,
        patience=4,
        batch_size=8,
        max_train=0,
        max_val=0,
        seed=42,
        pa_fraction=fraction,
        refresh_cache=False,
    )
    ckpt = Path(summary["checkpoint"]).resolve()
    if ckpt == SEED42.resolve():
        raise RuntimeError("refusing to overwrite the 50/50 checkpoint")
    eval_out = paper_runs() / f"mixed_imel_mix_{tag}_eval"
    print(f"===== score {label} on official eval lists =====", flush=True)
    metrics = eval_checkpoint_on_corpora(
        ckpt,
        asv17_root=asv17_root(),
        pa_root=pa_root(),
        asv17_split="eval",
        pa_split="eval",
        batch_size=16,
        output_dir=eval_out,
        refresh_cache=False,
    )
    pa_scores = eval_out / "pa2019_predictions.csv"
    row = {
        "pa_fraction": fraction,
        "label": label,
        "checkpoint": str(ckpt),
        "eer_2017": metrics["asvspoof2017"]["eer_percent"],
        "n_2017": metrics["asvspoof2017"]["n"],
        "eer_pa": metrics["pa2019"]["eer_percent"],
        "n_pa": metrics["pa2019"]["n"],
        "min_tdcf": min_tdcf_for_predictions(pa_scores),
        "trained_now": True,
    }
    rows.append(row)
    print(json.dumps(row, indent=2), flush=True)

rows.sort(key=lambda item: item["pa_fraction"], reverse=True)
table_path = paper_runs() / "mixed_imel_mix_ratios.json"
table_path.write_text(json.dumps(rows, indent=2), encoding="utf-8")
print("\n50/50 checkpoint stays at", SEED42)
print("Wrote", table_path)
print(f"{'Mix':<12} {'2017 EER':>10} {'PA EER':>10} {'min t-DCF':>10}")
for row in rows:
    print(
        f"{row['label']:<12} {row['eer_2017']:10.2f} {row['eer_pa']:10.2f} {row['min_tdcf']:10.4f}"
    )


PA dev probe PA_D_0000007 decodes, and 40/40 PA eval probes decode. 2017 eval audio is present.
No stale PA readability caches.

===== train inverted_mel 100% PA -> D:\speaker-verification-system\replay-cnn-baseline\experiments\lfcc_vs_mel_compare\runs\inverted_mel_mix_pa100 =====
Train inverted_mel | device=cuda | amp=True


Checking train audio: 0it [00:00, ?it/s]
Checking train audio: 0it [00:00, ?it/s]


Train: {"total": 86400, "asvspoof2017": 0, "pa2019": 86400, "bonafide": 8640, "spoof": 77760, "speakers": 16}
Val:   {"total": 11380, "asvspoof2017": 580, "pa2019": 10800, "bonafide": 1370, "spoof": 10010, "speakers": 6}


inverted_mel epoch 1/15:   0%|          | 0/10800 [00:00<?, ?it/s]

Scoring:   0%|          | 0/1423 [00:00<?, ?it/s]

...

epoch=6 train_loss=0.0004 val_eer=45.33% thr=0.0000
Early stopping
Done inverted_mel. Best mixed-val EER=44.53%
===== score 100% 2017 on official eval lists =====


Checking eval audio: 0it [00:00, ?it/s]


Scoring:   0%|          | 0/832 [00:00<?, ?it/s]

Scoring:   0%|          | 0/8421 [00:00<?, ?it/s]

{
  "pa_fraction": 0.0,
  "label": "100% 2017",
  "checkpoint": "D:\\speaker-verification-system\\replay-cnn-baseline\\experiments\\lfcc_vs_mel_compare\\runs\\inverted_mel_mix_pa000\\best_inverted_mel_mixed_2017_pa2019.pt",
  "eer_2017": 39.04620212103077,
  "n_2017": 13306,
  "eer_pa": 49.916959697397786,
  "n_pa": 134730,
  "min_tdcf": 0.9975377974483725,
  "trained_now": true
}

50/50 checkpoint stays at D:\speaker-verification-system\replay-cnn-baseline\experiments\lfcc_vs_mel_compare\runs\inverted_mel\best_inverted_mel_mixed_2017_pa2019.pt
Wrote D:\speaker-verification-system\replay-cnn-baseline\experiments\paper_official_eval\runs\mixed_imel_mix_ratios.json
Mix            2017 EER     PA EER  min t-DCF
100% PA           45.76       7.72     0.1908
75/25             26.19       8.67     0.2200
50/50             19.02       7.95     0.1984
25/75             24.44       8.77     0.2154
100% 2017         39.05      49.92     0.9975
